# CS231 - COMPUTER VISION

## Giới thiệu đồ án:
Dự án tập trung vào bài toán Fine-Grained Visual Categorization (FGVC) - phân loại chi tiết 150 loài chim trên tập dữ liệu Birds-525. Áp dụng và so sánh nhiều phương pháp từ truyền thống đến các kiến trúc SOTA (State-of-the-Art) như Bilinear CNN (B-CNN), Vision Transformer (ViT) và CLIP để tìm ra giải pháp tối ưu cho việc nhận diện các đặc trưng nhỏ giữa các loài chim có độ tương đồng cao.

## Thành viên:

| Thành viên | Vai trò | Nhiệm vụ trọng tâm |
| :--- | :--- | :--- |
| Member 1 | Kỹ sư Tiền xử lý & Baseline | Data Augmentation, HOG, VGG16 + Random Forest Baseline |
| Member 2 | Đánh giá & SOTA | Vision Transformer (ViT), Color Histogram, Evaluation Metrics |
| Member 3 | CNN & End-to-End | Custom Bilinear CNN (B-CNN), EfficientNetV2, End-to-End Training |
| Member 4 | Ứng dụng & CLIP | OpenAI-CLIP (Zero-shot), LBP Features, Gradio/Streamlit App |

# MEMBER 2:
- Công việc chung (Evaluation & Metrics): Viết script tính toán các độ đo: Accuracy, Precision,Recall,Macro F1.Vẽ Ma trận nhầm lẫn (Confusion Matrix) để phân tích lỗi sâu các loài chim hay bị nhầm với nhau.
- Trích xuất đặc trưng truyền thống: Cài đặt Color Histogram (trích xuất phân bố màu sắc lông chim).
- Trích xuất đặc trưng Học sâu: Cài đặt Vision Transformer (ViT-B/16) để trích xuất vector embedding 768 chiều.
- Mô hình phân loại: Random Forest (hoặc SVM) để phân loại dựa trên vector của ViT và Color Histogram.

# Rút trích đặc trưng

## Sử dụng Vision Transformer(ViT) để rút trích đặc trưng:

### Import Thư viện

In [39]:
import os
import numpy as np
import torch
from torchvision import datasets
from torch.utils.data import DataLoader, Dataset
from transformers import AutoImageProcessor, ViTModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from tqdm.notebook import tqdm
import warnings

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

warnings.filterwarnings("ignore")

### Thiết lập cấu hình và Kiểm tra GPU

In [41]:
DATA_DIR = '/kaggle/input/datasets/tonnguynb/augmented-dataset' 
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
VALID_DIR = os.path.join(DATA_DIR, 'val')
TEST_DIR = os.path.join(DATA_DIR, 'test')

# Kiểm tra GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang sử dụng thiết bị: {device}")

Đang sử dụng thiết bị: cuda


### Khởi tạo mô hình Vision Transformer (ViT)

In [42]:
MODEL_NAME = "google/vit-base-patch16-224-in21k"
print(f"Đang tải pre-trained model: {MODEL_NAME}...")

processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = ViTModel.from_pretrained(MODEL_NAME)

# Đưa model lên GPU và set chế độ evaluation
model = model.to(device)
model.eval() 
print("Tải model thành công!")

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


Đang tải pre-trained model: google/vit-base-patch16-224-in21k...


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Tải model thành công!


### Khai báo Dataset và DataLoaders

In [43]:
class BirdDataset(Dataset):
    def __init__(self, root_dir, processor):
        self.dataset = datasets.ImageFolder(root_dir)
        self.processor = processor

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        if image.mode != "RGB":
            image = image.convert("RGB")
            
        # Tự động resize và normalize theo chuẩn ViT
        pixel_values = self.processor(images=image, return_tensors="pt").pixel_values.squeeze()
        return pixel_values, label

BATCH_SIZE = 64

print("Đang cấu hình DataLoaders...")
train_dataset = BirdDataset(TRAIN_DIR, processor)
valid_dataset = BirdDataset(VALID_DIR, processor)
test_dataset = BirdDataset(TEST_DIR, processor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f"Số lượng class: {len(train_dataset.dataset.classes)}")

Đang cấu hình DataLoaders...
Số lượng class: 150


### Định nghĩa hàm trích xuất đặc trưng

In [44]:
def extract_features(dataloader, desc="Extracting"):
    features = []
    labels = []
    
    with torch.no_grad():
        for batch_images, batch_labels in tqdm(dataloader, desc=desc):
            batch_images = batch_images.to(device)
            
            outputs = model(pixel_values=batch_images)
            
            # Lấy đặc trưng từ token [CLS] (đại diện cho ảnh)
            cls_features = outputs.last_hidden_state[:, 0, :]
            
            features.append(cls_features.cpu().numpy())
            labels.extend(batch_labels.numpy())
            
    return np.vstack(features), np.array(labels)

### Thực thi trích xuất

In [45]:
print("BẮT ĐẦU TRÍCH XUẤT ĐẶC TRƯNG...")

X_train, y_train = extract_features(train_loader, desc="Train set")
X_valid, y_valid = extract_features(valid_loader, desc="Valid set")
X_test, y_test = extract_features(test_loader, desc="Test set")

print(f"\nKích thước tập Train: {X_train.shape}")
print(f"Kích thước tập Valid: {X_valid.shape}")
print(f"Kích thước tập Test : {X_test.shape}")

BẮT ĐẦU TRÍCH XUẤT ĐẶC TRƯNG...


Train set:   0%|          | 0/469 [00:00<?, ?it/s]

Valid set:   0%|          | 0/67 [00:00<?, ?it/s]

Test set:   0%|          | 0/70 [00:00<?, ?it/s]


Kích thước tập Train: (30000, 768)
Kích thước tập Valid: (4248, 768)
Kích thước tập Test : (4468, 768)


### Huấn luyện Random Forest

In [46]:
print("Đang huấn luyện Random Forest...")

# n_jobs=-1 giúp dùng full CPU của Kaggle
rf_classifier = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight='balanced')
rf_classifier.fit(X_train, y_train)

print("Huấn luyện hoàn tất!")

Đang huấn luyện Random Forest...
Huấn luyện hoàn tất!


### Đánh giá mô hình (Evaluation)

In [47]:
print("--- KẾT QUẢ ĐÁNH GIÁ ---")

# Validation
y_valid_pred = rf_classifier.predict(X_valid)
valid_acc = accuracy_score(y_valid, y_valid_pred)
print(f"Validation Accuracy: {valid_acc:.4f}")

# Testing
y_test_pred = rf_classifier.predict(X_test)
test_acc = accuracy_score(y_test, y_test_pred)
print(f"Test Accuracy:       {test_acc:.4f}")

# Mở comment đoạn dưới nếu bạn muốn xem chi tiết (Precision, Recall, F1) cho từng loài chim cụ thể
classes = train_dataset.dataset.classes
print("\nClassification Report trên tập Test:")
print(classification_report(y_test, y_test_pred, target_names=classes))

--- KẾT QUẢ ĐÁNH GIÁ ---
Validation Accuracy: 0.9706
Test Accuracy:       0.9696

Classification Report trên tập Test:
                               precision    recall  f1-score   support

                ABBOTTS BOOBY       0.93      0.97      0.95        29
   ABYSSINIAN GROUND HORNBILL       1.00      0.97      0.98        29
        AFRICAN PIED HORNBILL       1.00      1.00      1.00        30
          AFRICAN PYGMY GOOSE       0.93      0.93      0.93        29
                ALPINE CHOUGH       1.00      0.96      0.98        27
              AMERICAN AVOCET       1.00      1.00      1.00        29
             AMERICAN BITTERN       1.00      1.00      1.00        27
              AMERICAN DIPPER       1.00      1.00      1.00        30
               AMERICAN PIPIT       1.00      1.00      1.00        29
              AMERICAN WIGEON       1.00      1.00      1.00        30
            ASHY STORM PETREL       0.96      0.90      0.93        30
        ASIAN GREEN BEE EATE

In [49]:
import joblib

joblib.dump(
    rf_classifier,
    "random_forest.pkl"
)

print("RF model saved.")

RF model saved.


## Rút trích đặc trưng bằng Color Histogram

### Import Thư viện

In [52]:
import os
import cv2
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from tqdm.notebook import tqdm
import warnings

warnings.filterwarnings("ignore")

### Thiết lập cấu hình và Hàm trích đặc trưng Color Histogram

In [55]:
DATA_DIR = '/kaggle/input/datasets/tonnguynb/augmented-dataset' 
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
VALID_DIR = os.path.join(DATA_DIR, 'val') 
TEST_DIR = os.path.join(DATA_DIR, 'test')

# Hàm trích xuất Color Histogram cho 1 bức ảnh
def extract_color_histogram(image_path, bins=(8, 8, 8)):
    # Đọc ảnh bằng OpenCV (Mặc định đọc vào hệ màu BGR)
    image = cv2.imread(image_path)
    if image is None:
        return None
        
    # Chuyển đổi từ BGR sang HSV
    hsv_image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    
    # Tính toán Histogram
    # [0, 1, 2] là 3 kênh H, S, V
    # ranges: H từ 0-180, S và V từ 0-256 (chuẩn của OpenCV)
    hist = cv2.calcHist([hsv_image], [0, 1, 2], None, bins, [0, 180, 0, 256, 0, 256])
    
    # Chuẩn hóa Histogram (Rất quan trọng!)
    # Giúp đặc trưng không bị ảnh hưởng bởi kích thước to/nhỏ của bức ảnh
    cv2.normalize(hist, hist)
    
    # Kéo phẳng mảng 3D thành vector 1D (Ví dụ: 8x8x8 = 512 chiều)
    return hist.flatten()

### Hàm duyệt qua thư mục và nạp dữ liệu

In [56]:
def load_dataset_with_histogram(folder_path, bins=(8, 8, 8)):
    features = []
    labels = []
    
    # Lấy danh sách tên class từ tên thư mục
    classes = sorted(os.listdir(folder_path))
    class_to_idx = {cls_name: idx for idx, cls_name in enumerate(classes)}
    
    for cls_name in tqdm(classes, desc=f"Đang xử lý {os.path.basename(folder_path)}"):
        class_dir = os.path.join(folder_path, cls_name)
        if not os.path.isdir(class_dir):
            continue
            
        for img_name in os.listdir(class_dir):
            img_path = os.path.join(class_dir, img_name)
            
            # Trích xuất đặc trưng
            hist_feature = extract_color_histogram(img_path, bins)
            
            if hist_feature is not None:
                features.append(hist_feature)
                labels.append(class_to_idx[cls_name])
                
    return np.array(features), np.array(labels), classes

print("--- BẮT ĐẦU TRÍCH XUẤT COLOR HISTOGRAM ---")
# Số bins=(8,8,8) sẽ tạo ra vector 512 chiều. Bạn có thể thử (16,16,16) để lấy chi tiết màu hơn (ra 4096 chiều).
X_train, y_train, classes = load_dataset_with_histogram(TRAIN_DIR, bins=(8, 8, 8))
X_valid, y_valid, _ = load_dataset_with_histogram(VALID_DIR, bins=(8, 8, 8))
X_test, y_test, _ = load_dataset_with_histogram(TEST_DIR, bins=(8, 8, 8))

print(f"\nKích thước tập Train: {X_train.shape}")

--- BẮT ĐẦU TRÍCH XUẤT COLOR HISTOGRAM ---


Đang xử lý train:   0%|          | 0/150 [00:00<?, ?it/s]

Đang xử lý val:   0%|          | 0/150 [00:00<?, ?it/s]

Đang xử lý test:   0%|          | 0/150 [00:00<?, ?it/s]


Kích thước tập Train: (30000, 512)


### Train Random Forest

In [58]:
print("Đang huấn luyện Random Forest...")

# class_weight='balanced' giúp xử lý nếu số lượng ảnh giữa các loài bị lệch
rf_color = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight='balanced')
rf_color.fit(X_train, y_train)

print("Huấn luyện hoàn tất!\n")

Đang huấn luyện Random Forest...
Huấn luyện hoàn tất!



### Đánh giá

In [59]:
print("--- KẾT QUẢ ĐÁNH GIÁ (COLOR HISTOGRAM) ---")
# Validation
y_valid_pred = rf_color.predict(X_valid)
print(f"Validation Accuracy: {accuracy_score(y_valid, y_valid_pred):.4f}")

# Testing
y_test_pred = rf_color.predict(X_test)
print(f"Test Accuracy:       {accuracy_score(y_test, y_test_pred):.4f}")

--- KẾT QUẢ ĐÁNH GIÁ (COLOR HISTOGRAM) ---
Validation Accuracy: 0.3708
Test Accuracy:       0.3688


In [61]:
import joblib

joblib.dump(
    rf_color,
    "rf_color.pkl"
)

['rf_color.pkl']